In [2]:
import os
from utils.data_utils import load_metadata_and_map
from cuml.cluster import KMeans
import numpy as np
import cupy as cp
import torch
import pandas as pd
from utils import utils
from utils.utils import load_tensor

In [3]:
METADATA_FILE = "embeddings/id_metadata.json"
TENSOR_DIRECTORY = "embeddings/uuid_embeddings/embeddings/"

In [4]:
tensors = load_tensor(TENSOR_DIRECTORY, num_workers=16)

Loading cache from embeddings/cache


In [5]:
metadata_df, artist_to_track_idx = load_metadata_and_map(METADATA_FILE, tensors)

Loading metadata from embeddings/id_metadata.json...
Building Artist to Track Index map...


In [6]:
all_embeddings = utils.pool_loaded_tensor_dict(tensors=tensors, mode='mean+max')

embeddings_torch = torch.stack(list(all_embeddings.values())).cuda()

embeddings_gpu = cp.asarray(embeddings_torch)

Detected allchunks tensors, pooling to 1D embedding (mean+max)


100%|██████████| 138748/138748 [00:02<00:00, 47163.68it/s]


In [12]:
import json
from typing import Dict
from utils.data_utils import get_artist_name

artist_variance: Dict[str, float] = {}
for artist_id, track_idx in artist_to_track_idx.items():
  track_idx = cp.asarray(track_idx)
  # skip artists with less than 10
  if len(track_idx) < 40:
    continue

  track_embeddings = embeddings_gpu[track_idx]
  # Compute the variance of the track embeddings
  track_variance = cp.var(track_embeddings, axis=0)
  # total variance of the artist
  artist_variance[artist_id] = cp.sum(track_variance)

# Sort artists by variance
sorted_artists = sorted(artist_variance.items(), key=lambda x: x[1], reverse=True)

top = sorted_artists[:1000]

# export list to csv
artist_id_names = [{'Id': artist_id, 'Name': get_artist_name(metadata_df, artist_id)} for artist_id, _ in top]

# export to json
with open('data/validation/top_1000_artists_by_variance.json', 'w') as f:
  json.dump(artist_id_names, f)

